In [1]:
import pandas as pd
from pandas import DataFrame
import os
import folium
import sqlite3

In [2]:
import sqlite3
import pandas as pd

# 1. Connect to the SQLite database
with sqlite3.connect("../data/processed/flight_data.db") as conn:
    # 2. Run a query and load directly into a DataFrame
    query = 'SELECT name, iata_code, latitude_deg, longitude_deg FROM airports_our_airports WHERE iso_country = "US" and type = "large_airport"'
    airports_df = pd.read_sql_query(query, conn)
airports_df.head(20)

,name,iata_code,latitude_deg,longitude_deg
0,Albuquerque International Sunport,ABQ,35.039976,-106.608925
1,Albany International Airport,ALB,42.748299,-73.801697
2,Hartsfield Jackson Atlanta International Airport,ATL,33.636700,-84.428101
3,Austin Bergstrom International Airport,AUS,30.197535,-97.662015
4,Bradley International Airport,BDL,41.938555,-72.688016
5,Birmingham-Shuttlesworth International Airport,BHM,33.562877,-86.750712
6,Nashville International Airport,BNA,36.124500,-86.678200
7,Boise Air Terminal/Gowen Field,BOI,43.564400,-116.223000
8,Boston Logan International Airport,BOS,42.361970,-71.007900
9,Buffalo Niagara International Airport,BUF,42.940498,-78.732201


In [3]:
map = folium.Map(location=[44.9672, -103.7716], zoom_start=3)

In [4]:
def apply_to_each_row(row):
    iata = row["iata_code"]
    lat = row["latitude_deg"]
    long = row["longitude_deg"]
    name = row["name"]
    if iata is not None and lat is not None and long is not None and name is not None:
        folium.Marker(
            location=[float(lat), float(long)],
            popup=iata,
            tooltip=name,
            icon=folium.Icon(color="blue", icon="info-sign")
).add_to(map)

airports_df.apply(apply_to_each_row, axis=1)
map

In [5]:
import folium
from pyproj import Geod

# Define endpoints: [latitude, longitude]
boston = [42.3581, -71.0636]
sf = [37.7833, -122.4167]

# Initialize WGS84 geodesic model
geod = Geod(ellps="WGS84")

# pyproj uses (lon, lat) order for npts
# npts calculates N intermediate points between start and end
num_intermediate_points = 13
points = geod.npts(
    lon1=boston[1],
    lat1=boston[0],
    lon2=sf[1],
    lat2=sf[0],
    npts=num_intermediate_points,
)

# Combine start, intermediate points, and end into [lat, lon] format for Folium
coordinates = [boston] + [[lat, lon] for lon, lat in points] + [sf]

# Create Folium Map
m = folium.Map(location=[41.9, -97.3], zoom_start=4)

folium.PolyLine(
    locations=coordinates,
    color="#FF0000",
    weight=5,
    tooltip="From Boston to San Francisco",
).add_to(m)

# Save or display map
# m.save("great_circle_map.html")

ModuleNotFoundError: No module named 'pyproj'